# Fake News Detection — WELFake + GloVe LSTM + LIME
**Dataset:** WELFake (72k articles, 4 sources merged)  
**Model:** LSTM with GloVe-100d pretrained embeddings  
**Explainability:** LIME word highlights  
**Goal:** PPT-ready graphs + under 10 min training on Colab T4

## Cell 1 — Install & imports

In [ ]:
# Install dependencies
!pip install lime kaggle --quiet

import os, re, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from lime.lime_text import LimeTextExplainer

# ── PPT-ready plot style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 13,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
})

PALETTE = {'real': '#2ecc71', 'fake': '#e74c3c',
           'blue': '#3498db', 'purple': '#9b59b6',
           'orange': '#e67e22', 'dark': '#2c3e50'}

print('✅ All imports done. GPU:', tf.config.list_physical_devices('GPU'))

## Cell 2 — Download WELFake dataset from Kaggle

In [ ]:
# ── Option A: Upload kaggle.json (recommended) ────────────────────────
# 1. Go to kaggle.com → Account → API → Create New Token → downloads kaggle.json
# 2. Run the cell below to upload it

from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d saurabhshahane/fake-news-classification -p /content/
!unzip -o /content/fake-news-classification.zip -d /content/welface/
print('✅ WELFake downloaded')

In [ ]:
# ── Option B: Manual upload (if no Kaggle account) ───────────────────
# Download WELFake.csv from:
# https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification
# Then upload it here:

# from google.colab import files
# uploaded = files.upload()  # upload WELFake.csv directly

## Cell 3 — Load & explore data

In [ ]:
# Load — adjust path if you used Option B
df = pd.read_csv('/content/welface/WELFake_Dataset.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nLabel distribution:')
print(df['label'].value_counts())
print('\nSample:')
df.head(3)

In [ ]:
# ── Graph 1: Class distribution (PPT slide 1) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
counts = df['label'].value_counts().sort_index()
labels_text = ['Fake', 'Real']
colors = [PALETTE['fake'], PALETTE['real']]
bars = axes[0].bar(labels_text, counts.values, color=colors,
                   width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 300,
                 f'{val:,}', ha='center', va='bottom',
                 fontweight='bold', fontsize=13)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Number of Articles')
axes[0].set_ylim(0, counts.max() * 1.15)

# Pie chart
axes[1].pie(counts.values, labels=labels_text, colors=colors,
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 13},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Label Split')

plt.suptitle('WELFake Dataset Overview', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('graph_01_class_distribution.png')
plt.show()
print('✅ Saved: graph_01_class_distribution.png')

In [ ]:
# ── Graph 2: Article length distribution (PPT slide 2) ────────────────
df['text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(11, 5))

for lbl, color, name in [(0, PALETTE['fake'], 'Fake'),
                          (1, PALETTE['real'], 'Real')]:
    data = df[df['label'] == lbl]['word_count'].clip(upper=1500)
    ax.hist(data, bins=60, alpha=0.6, color=color,
            label=name, edgecolor='white')

ax.axvline(200, color=PALETTE['dark'], linestyle='--', linewidth=1.5,
           label='maxlen=200 (model cutoff)')
ax.set_xlabel('Word Count per Article')
ax.set_ylabel('Number of Articles')
ax.set_title('Article Length Distribution')
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('graph_02_length_distribution.png')
plt.show()
print('✅ Saved: graph_02_length_distribution.png')

## Cell 4 — Text preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-zA-Z ]', '', text)         # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Drop rows with empty text
df = df.dropna(subset=['text', 'label']).reset_index(drop=True)
df['clean'] = df['text'].apply(clean_text)
df = df[df['clean'].str.len() > 20].reset_index(drop=True)

print(f'Articles after cleaning: {len(df):,}')
print('Sample cleaned text:')
print(df['clean'].iloc[0][:200])

In [ ]:
# ── Tokenize & pad ────────────────────────────────────────────────────
VOCAB_SIZE = 20000   # larger vocab for WELFake's diversity
MAX_LEN    = 300     # longer than ISOT since articles are more varied

X = df['clean'].values
y = df['label'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_raw)

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = encode(X_train_raw)
X_test  = encode(X_test_raw)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Vocab size (actual): {len(tokenizer.word_index):,}')

## Cell 5 — Load GloVe embeddings

In [ ]:
# Download GloVe 100d (~330 MB) — takes ~2 min on Colab
!wget -q --show-progress https://nlp.stanford.edu/data/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.100d.txt
print('✅ GloVe downloaded')

In [ ]:
EMBED_DIM = 100

print('Loading GloVe vectors...')
glove = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        vals = line.split()
        glove[vals[0]] = np.array(vals[1:], dtype='float32')

print(f'GloVe loaded: {len(glove):,} word vectors')

# Build embedding matrix
word_index = tokenizer.word_index
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
hits, misses = 0, 0

for word, i in word_index.items():
    if i >= VOCAB_SIZE:
        continue
    vec = glove.get(word)
    if vec is not None:
        embedding_matrix[i] = vec
        hits += 1
    else:
        misses += 1

print(f'Embedding matrix: {embedding_matrix.shape}')
print(f'Words found in GloVe: {hits:,} | Not found: {misses:,} ({misses/(hits+misses)*100:.1f}%)')

## Cell 6 — Build & train LSTM model

In [ ]:
# ── Model architecture ────────────────────────────────────────────────
model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True          # fine-tune GloVe on this corpus
    ),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
], name='FakeNews_BiLSTM')

model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────
# On Colab T4: ~4-7 min for 5 epochs at batch_size=256
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=2,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=256,       # large batch = faster on GPU
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print('\n✅ Training complete')

## Cell 7 — Evaluation & PPT graphs

In [ ]:
# Predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype('int32').flatten()

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1 Score:  {f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

In [ ]:
# ── Graph 3: Training curves (PPT slide 3) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

epochs_ran = range(1, len(history.history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs_ran, history.history['accuracy'],
             color=PALETTE['blue'], linewidth=2.5,
             marker='o', markersize=6, label='Training')
axes[0].plot(epochs_ran, history.history['val_accuracy'],
             color=PALETTE['orange'], linewidth=2.5,
             marker='s', markersize=6, linestyle='--', label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.5, 1.02)
axes[0].legend(fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

# Loss
axes[1].plot(epochs_ran, history.history['loss'],
             color=PALETTE['blue'], linewidth=2.5,
             marker='o', markersize=6, label='Training')
axes[1].plot(epochs_ran, history.history['val_loss'],
             color=PALETTE['orange'], linewidth=2.5,
             marker='s', markersize=6, linestyle='--', label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=12)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Training History — BiLSTM + GloVe', fontsize=16,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('graph_03_training_curves.png')
plt.show()
print('✅ Saved: graph_03_training_curves.png')

In [ ]:
# ── Graph 4: Confusion matrix (PPT slide 4) ───────────────────────────
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7, 6))

# Custom color matrix
cm_display = np.array([[tn, fp], [fn, tp]])
im = ax.imshow(cm_display, interpolation='nearest',
               cmap='RdYlGn', aspect='auto', vmin=0)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

tick_labels = ['Fake (0)', 'Real (1)']
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(tick_labels, fontsize=13)
ax.set_yticklabels(tick_labels, fontsize=13)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Confusion Matrix', fontsize=16, fontweight='bold', pad=15)

# Annotate cells
total = cm_display.sum()
for i in range(2):
    for j in range(2):
        val = cm_display[i, j]
        pct = val / total * 100
        ax.text(j, i, f'{val:,}\n({pct:.1f}%)',
                ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if val < cm_display.max() * 0.6 else 'black')

plt.tight_layout()
plt.savefig('graph_04_confusion_matrix.png')
plt.show()
print('✅ Saved: graph_04_confusion_matrix.png')

In [ ]:
# ── Graph 5: Metrics bar chart (PPT slide 5) ──────────────────────────
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1}

fig, ax = plt.subplots(figsize=(9, 5))

bar_colors = [PALETTE['blue'], PALETTE['purple'],
              PALETTE['orange'], PALETTE['real']]
bars = ax.bar(metrics.keys(), metrics.values(),
              color=bar_colors, width=0.55,
              edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom',
            fontweight='bold', fontsize=13)

ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Performance — BiLSTM + GloVe on WELFake',
             fontsize=15, fontweight='bold')
ax.axhline(y=0.95, color='gray', linestyle=':', linewidth=1,
           label='0.95 reference line')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('graph_05_metrics.png')
plt.show()
print('✅ Saved: graph_05_metrics.png')

In [ ]:
# ── Graph 6: Prediction confidence histogram (PPT slide 6) ────────────
fig, ax = plt.subplots(figsize=(10, 5))

fake_probs = y_pred_prob[y_test == 0].flatten()
real_probs = y_pred_prob[y_test == 1].flatten()

ax.hist(fake_probs, bins=50, alpha=0.7, color=PALETTE['fake'],
        label='True Fake articles', edgecolor='white')
ax.hist(real_probs, bins=50, alpha=0.7, color=PALETTE['real'],
        label='True Real articles', edgecolor='white')
ax.axvline(0.5, color=PALETTE['dark'], linewidth=2,
           linestyle='--', label='Decision threshold (0.5)')

ax.set_xlabel('Predicted Probability of Being Real')
ax.set_ylabel('Number of Articles')
ax.set_title('Model Confidence Distribution', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('graph_06_confidence_distribution.png')
plt.show()
print('✅ Saved: graph_06_confidence_distribution.png')

## Cell 8 — LIME explainability (PPT-ready word highlights)

In [ ]:
# ── LIME predict function ─────────────────────────────────────────────
def predict_proba_lstm(texts):
    """Wrapper for LIME: text list → (N, 2) probability array."""
    seqs   = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(seqs, maxlen=MAX_LEN,
                           padding='post', truncating='post')
    probs  = model.predict(padded, verbose=0, batch_size=64)
    return np.hstack([1 - probs, probs])   # [P(fake), P(real)]

explainer = LimeTextExplainer(
    class_names=['Fake', 'Real'],
    random_state=42
)

print('✅ LIME explainer ready')

In [ ]:
# ── Graph 7 & 8: LIME word importance — one fake, one real article ─────

def plot_lime_explanation(idx, split='test', num_features=12,
                          num_samples=2000, save_name=None):
    """Generate a clean horizontal bar chart of LIME word weights."""
    if split == 'test':
        text  = X_test_raw[idx]
        label = y_test[idx]
    else:
        text  = X_train_raw[idx]
        label = y_train[idx]

    true_name = 'Real' if label == 1 else 'Fake'
    pred_prob = model.predict(encode([text]), verbose=0)[0][0]
    pred_name = 'Real' if pred_prob > 0.5 else 'Fake'
    confidence = pred_prob if pred_prob > 0.5 else 1 - pred_prob

    exp = explainer.explain_instance(
        text,
        predict_proba_lstm,
        num_features=num_features,
        num_samples=num_samples,
        labels=[1]
    )

    word_weights = exp.as_list(label=1)   # positive = pushes toward Real
    words  = [w for w, _ in word_weights]
    weights = [v for _, v in word_weights]

    # Sort by weight
    sorted_pairs = sorted(zip(words, weights), key=lambda x: x[1])
    words   = [p[0] for p in sorted_pairs]
    weights = [p[1] for p in sorted_pairs]
    colors  = [PALETTE['real'] if w > 0 else PALETTE['fake'] for w in weights]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(words, weights, color=colors,
                   edgecolor='white', linewidth=0.8, height=0.65)

    ax.axvline(0, color=PALETTE['dark'], linewidth=1.2)
    ax.set_xlabel('LIME Weight (positive = pushes toward Real)', fontsize=12)
    ax.set_title(
        f'LIME Explanation\nTrue: {true_name}  |  Predicted: {pred_name}  '
        f'({confidence*100:.1f}% confidence)',
        fontsize=14, fontweight='bold'
    )

    # Value labels on bars
    for bar, val in zip(bars, weights):
        xpos = bar.get_width() + 0.003 if val >= 0 else bar.get_width() - 0.003
        ha   = 'left' if val >= 0 else 'right'
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', ha=ha, fontsize=10)

    # Legend patches
    ax.legend(
        handles=[
            mpatches.Patch(color=PALETTE['real'], label='Pushes → Real'),
            mpatches.Patch(color=PALETTE['fake'], label='Pushes → Fake')
        ],
        loc='lower right', fontsize=11
    )
    ax.grid(axis='x', alpha=0.25)

    plt.tight_layout()
    if save_name:
        plt.savefig(save_name)
        print(f'✅ Saved: {save_name}')
    plt.show()

# Find a correctly predicted fake article
fake_idxs = np.where((y_test == 0) & (y_pred == 0))[0]
real_idxs = np.where((y_test == 1) & (y_pred == 1))[0]

print('Explaining a FAKE article...')
plot_lime_explanation(fake_idxs[3], num_features=12, num_samples=2000,
                      save_name='graph_07_lime_fake_article.png')

print('Explaining a REAL article...')
plot_lime_explanation(real_idxs[3], num_features=12, num_samples=2000,
                      save_name='graph_08_lime_real_article.png')

In [ ]:
# ── Graph 9: LIME notebook HTML highlight (inline, best for Colab) ─────
# This shows the coloured word-in-sentence highlight view

idx  = fake_idxs[3]
text = X_test_raw[idx]

exp = explainer.explain_instance(
    text,
    predict_proba_lstm,
    num_features=15,
    num_samples=2000,
    labels=[1]
)

# Renders coloured inline HTML in Colab
exp.show_in_notebook(text=True)

# Save as standalone HTML (can screenshot for PPT)
with open('graph_09_lime_highlight.html', 'w') as f:
    f.write(exp.as_html())
print('✅ Saved: graph_09_lime_highlight.html  (screenshot this for PPT)')

In [ ]:
# ── Graph 10: Aggregate word importance across N articles (PPT slide 9) 
# Shows overall pattern, not just one article

N_ARTICLES = 30   # explain 30 articles — ~3 min on Colab
all_weights = {}

sample_fake = fake_idxs[:N_ARTICLES // 2]
sample_real = real_idxs[:N_ARTICLES // 2]
sample_idxs = np.concatenate([sample_fake, sample_real])

print(f'Running LIME on {N_ARTICLES} articles... (~3 min)')
for i, idx in enumerate(sample_idxs):
    if i % 5 == 0:
        print(f'  {i}/{N_ARTICLES}...')
    text = X_test_raw[idx]
    try:
        exp = explainer.explain_instance(
            text, predict_proba_lstm,
            num_features=10, num_samples=1000, labels=[1]
        )
        for word, weight in exp.as_list(label=1):
            word = word.strip().lower()
            if word not in all_weights:
                all_weights[word] = []
            all_weights[word].append(weight)
    except Exception:
        pass

# Aggregate: mean weight, only words seen in 3+ articles
agg = {w: np.mean(v) for w, v in all_weights.items() if len(v) >= 3}
agg_sorted = sorted(agg.items(), key=lambda x: abs(x[1]), reverse=True)[:20]

words_agg   = [p[0] for p in reversed(agg_sorted)]
weights_agg = [p[1] for p in reversed(agg_sorted)]
colors_agg  = [PALETTE['real'] if w > 0 else PALETTE['fake']
               for w in weights_agg]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(words_agg, weights_agg, color=colors_agg,
        edgecolor='white', linewidth=0.8, height=0.7)
ax.axvline(0, color=PALETTE['dark'], linewidth=1.2)
ax.set_xlabel('Mean LIME Weight across articles', fontsize=12)
ax.set_title(
    f'Top 20 Most Influential Words\n(aggregated over {N_ARTICLES} articles)',
    fontsize=15, fontweight='bold'
)
ax.legend(
    handles=[
        mpatches.Patch(color=PALETTE['real'], label='Pushes → Real'),
        mpatches.Patch(color=PALETTE['fake'], label='Pushes → Fake')
    ],
    fontsize=11
)
ax.grid(axis='x', alpha=0.25)

plt.tight_layout()
plt.savefig('graph_10_lime_aggregate.png')
plt.show()
print('✅ Saved: graph_10_lime_aggregate.png')

## Cell 9 — Download all graphs

In [ ]:
# Zip all saved graphs and download
import zipfile
from google.colab import files

graph_files = [
    'graph_01_class_distribution.png',
    'graph_02_length_distribution.png',
    'graph_03_training_curves.png',
    'graph_04_confusion_matrix.png',
    'graph_05_metrics.png',
    'graph_06_confidence_distribution.png',
    'graph_07_lime_fake_article.png',
    'graph_08_lime_real_article.png',
    'graph_09_lime_highlight.html',
    'graph_10_lime_aggregate.png',
]

with zipfile.ZipFile('ppt_graphs.zip', 'w') as z:
    for fname in graph_files:
        if os.path.exists(fname):
            z.write(fname)
            print(f'  + {fname}')
        else:
            print(f'  ! Missing: {fname}')

files.download('ppt_graphs.zip')
print('\n✅ All graphs downloaded as ppt_graphs.zip')

## Summary of saved graphs

| File | What it shows | Suggested PPT slide |
|------|--------------|--------------------|
| graph_01_class_distribution.png | Dataset balance | Dataset intro |
| graph_02_length_distribution.png | Article length vs maxlen cutoff | Preprocessing |
| graph_03_training_curves.png | Accuracy & loss per epoch | Model training |
| graph_04_confusion_matrix.png | TP/TN/FP/FN breakdown | Results |
| graph_05_metrics.png | Accuracy/Precision/Recall/F1 bars | Results summary |
| graph_06_confidence_distribution.png | Model confidence per class | Results depth |
| graph_07_lime_fake_article.png | LIME — why model said Fake | Explainability |
| graph_08_lime_real_article.png | LIME — why model said Real | Explainability |
| graph_09_lime_highlight.html | Coloured word highlight in article | Explainability demo |
| graph_10_lime_aggregate.png | Top words across 30 articles | Explainability summary |